# Backtest Data Readiness Check

Checks all data required by backtest CLI:
- `market_snapshot` — pre-filter candidates selection
- `trades` — tick data for factor computation
- `quotes` — quote data for spread factors
- `bars_1m` — 1-minute bars for bar-based factors

Outputs a readiness report showing which dates are ready for backtest.

In [1]:
import os
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import clickhouse_connect
import polars as pl

ET = ZoneInfo("America/New_York")

ch = clickhouse_connect.get_client(
    host="localhost",
    port=8123,
    username="default",
    password=os.getenv("CLICKHOUSE_PASSWORD", ""),
    database="jerry_trader",
)
print("ClickHouse connected")

ClickHouse connected


## Configuration

In [2]:
START_DATE = "2026-03-02"
END_DATE = "2026-03-15"

start = datetime.strptime(START_DATE, "%Y-%m-%d")
end = datetime.strptime(END_DATE, "%Y-%m-%d")
dates_to_check = []
current = start
while current <= end:
    dates_to_check.append(current.strftime("%Y-%m-%d"))
    current += timedelta(days=1)

print(f"Checking {len(dates_to_check)} dates: {START_DATE} to {END_DATE}")

Checking 14 dates: 2026-03-02 to 2026-03-15


## 1. Table Existence

In [3]:
REQUIRED_TABLES = ["market_snapshot", "trades", "quotes", "bars_1m"]

existing_tables = [r[0] for r in ch.query("SHOW TABLES").result_rows]
missing = [t for t in REQUIRED_TABLES if t not in existing_tables]

if missing:
    print(f"MISSING TABLES: {missing}")
else:
    print(f"All required tables exist: {REQUIRED_TABLES}")

MISSING TABLES: ['bars_1m']


## 2. Per-Date Data Availability

In [4]:
def check_date_data(date: str) -> dict:
    result = {"date": date, "tables": {}, "ready": True, "issues": []}
    
    # market_snapshot
    try:
        cnt = ch.query(
            "SELECT count() FROM market_snapshot FINAL WHERE date = {d:String}",
            parameters={"d": date}
        ).result_rows[0][0]
        tickers = ch.query(
            "SELECT count(DISTINCT symbol) FROM market_snapshot FINAL WHERE date = {d:String}",
            parameters={"d": date}
        ).result_rows[0][0]
        result["tables"]["market_snapshot"] = {
            "rows": cnt, "tickers": tickers, "status": "OK" if cnt > 0 else "MISSING"
        }
        if cnt == 0:
            result["ready"] = False
            result["issues"].append("market_snapshot: no data")
    except Exception as e:
        result["tables"]["market_snapshot"] = {"rows": 0, "status": "ERROR"}
        result["ready"] = False
        result["issues"].append("market_snapshot: query failed")
    
    # trades
    try:
        cnt = ch.query(
            "SELECT count() FROM trades FINAL WHERE date = {d:String}",
            parameters={"d": date}
        ).result_rows[0][0]
        tickers = ch.query(
            "SELECT count(DISTINCT ticker) FROM trades FINAL WHERE date = {d:String}",
            parameters={"d": date}
        ).result_rows[0][0]
        result["tables"]["trades"] = {
            "rows": cnt, "tickers": tickers, "status": "OK" if cnt > 0 else "MISSING"
        }
        if cnt == 0:
            result["ready"] = False
            result["issues"].append("trades: no tick data - run import-ticks")
    except Exception as e:
        result["tables"]["trades"] = {"rows": 0, "status": "ERROR"}
        result["ready"] = False
        result["issues"].append("trades: query failed")
    
    # quotes
    try:
        cnt = ch.query(
            "SELECT count() FROM quotes FINAL WHERE date = {d:String}",
            parameters={"d": date}
        ).result_rows[0][0]
        tickers = ch.query(
            "SELECT count(DISTINCT ticker) FROM quotes FINAL WHERE date = {d:String}",
            parameters={"d": date}
        ).result_rows[0][0]
        result["tables"]["quotes"] = {
            "rows": cnt, "tickers": tickers, "status": "OK" if cnt > 0 else "MISSING"
        }
        if cnt == 0:
            result["ready"] = False
            result["issues"].append("quotes: no tick data - run import-ticks")
    except Exception as e:
        result["tables"]["quotes"] = {"rows": 0, "status": "ERROR"}
        result["ready"] = False
        result["issues"].append("quotes: query failed")
    
    # bars_1m
    try:
        cnt = ch.query(
            "SELECT count() FROM bars_1m FINAL WHERE date = {d:String}",
            parameters={"d": date}
        ).result_rows[0][0]
        tickers = ch.query(
            "SELECT count(DISTINCT symbol) FROM bars_1m FINAL WHERE date = {d:String}",
            parameters={"d": date}
        ).result_rows[0][0]
        result["tables"]["bars_1m"] = {
            "rows": cnt, "tickers": tickers, "status": "OK" if cnt > 0 else "MISSING"
        }
        if cnt == 0:
            result["ready"] = False
            result["issues"].append("bars_1m: no data")
    except Exception as e:
        result["tables"]["bars_1m"] = {"rows": 0, "status": "ERROR"}
        result["ready"] = False
        result["issues"].append("bars_1m: query failed")
    
    return result

results = [check_date_data(d) for d in dates_to_check]
print(f"Checked {len(results)} dates")

Checked 14 dates


In [5]:
rows = []
for r in results:
    status_icon = "OK" if r["ready"] else "MISSING"
    snapshot = r["tables"].get("market_snapshot", {})
    trades = r["tables"].get("trades", {})
    quotes = r["tables"].get("quotes", {})
    bars = r["tables"].get("bars_1m", {})
    
    rows.append({
        "date": r["date"],
        "status": status_icon,
        "snapshot_tickers": snapshot.get("tickers", 0),
        "trades_tickers": trades.get("tickers", 0),
        "quotes_tickers": quotes.get("tickers", 0),
        "bars_tickers": bars.get("tickers", 0),
        "issues": " | " .join(r["issues"]) if r["issues"] else "",
    })

pl.DataFrame(rows)

date,status,snapshot_tickers,trades_tickers,quotes_tickers,bars_tickers,issues
str,str,i64,i64,i64,i64,str
"""2026-03-02""","""MISSING""",76,76,76,0,"""bars_1m: query failed"""
"""2026-03-03""","""MISSING""",104,104,104,0,"""bars_1m: query failed"""
"""2026-03-04""","""MISSING""",134,134,134,0,"""bars_1m: query failed"""
"""2026-03-05""","""MISSING""",120,120,120,0,"""bars_1m: query failed"""
"""2026-03-06""","""MISSING""",131,131,131,0,"""bars_1m: query failed"""
…,…,…,…,…,…,…
"""2026-03-11""","""MISSING""",0,0,0,0,"""market_snapshot: no data | tra…"
"""2026-03-12""","""MISSING""",0,0,0,0,"""market_snapshot: no data | tra…"
"""2026-03-13""","""MISSING""",117,117,117,0,"""bars_1m: query failed"""


## 3. Readiness Summary

In [6]:
ready_dates = [r for r in results if r["ready"]]
not_ready = [r for r in results if not r["ready"]]

print(f"READY: {len(ready_dates)}/{len(results)} dates")
print(f"Ready dates: {[r['date'] for r in ready_dates]}")

if not_ready:
    print(f"\nNOT READY ({len(not_ready)} dates):")
    for r in not_ready:
        print(f"  {r['date']}: {', '.join(r['issues'])}")

READY: 0/14 dates
Ready dates: []

NOT READY (14 dates):
  2026-03-02: bars_1m: query failed
  2026-03-03: bars_1m: query failed
  2026-03-04: bars_1m: query failed
  2026-03-05: bars_1m: query failed
  2026-03-06: bars_1m: query failed
  2026-03-07: market_snapshot: no data, trades: no tick data - run import-ticks, quotes: no tick data - run import-ticks, bars_1m: query failed
  2026-03-08: market_snapshot: no data, trades: no tick data - run import-ticks, quotes: no tick data - run import-ticks, bars_1m: query failed
  2026-03-09: market_snapshot: no data, trades: no tick data - run import-ticks, quotes: no tick data - run import-ticks, bars_1m: query failed
  2026-03-10: market_snapshot: no data, trades: no tick data - run import-ticks, quotes: no tick data - run import-ticks, bars_1m: query failed
  2026-03-11: market_snapshot: no data, trades: no tick data - run import-ticks, quotes: no tick data - run import-ticks, bars_1m: query failed
  2026-03-12: market_snapshot: no data, tra

## 4. Recommended Actions

In [7]:
missing_snapshot = [r for r in not_ready if "market_snapshot" in str(r["issues"])]
missing_tickdata = [r for r in not_ready if "trades" in str(r["issues"]) or "quotes" in str(r["issues"])]
missing_bars = [r for r in not_ready if "bars_1m" in str(r["issues"])]

print("=== RECOMMENDED ACTIONS ===\n")

if missing_snapshot:
    dates = [r['date'] for r in missing_snapshot]
    print(f"1. Build market_snapshot for {len(dates)} dates:")
    print(f"   poetry run python -m jerry_trader.services.backtest.data.cli build-snapshot --date {dates[0]} --reprocess --force")

if missing_tickdata:
    dates = [r['date'] for r in missing_tickdata]
    print(f"2. Import tickdata for {len(dates)} dates:")
    print(f"   poetry run python -m jerry_trader.services.backtest.data.cli import-ticks --date {dates[0]}")

if missing_bars:
    dates = [r['date'] for r in missing_bars]
    print(f"3. Build bars_1m for {len(dates)} dates")

if not missing_snapshot and not missing_tickdata and not missing_bars:
    print("All dates ready! Run backtest:")
    print(f"   poetry run python -m jerry_trader.services.backtest.cli --date {ready_dates[0]['date']}")

=== RECOMMENDED ACTIONS ===

1. Build market_snapshot for 8 dates:
   poetry run python -m jerry_trader.services.backtest.data.cli build-snapshot --date 2026-03-07 --reprocess --force
2. Import tickdata for 8 dates:
   poetry run python -m jerry_trader.services.backtest.data.cli import-ticks --date 2026-03-07
3. Build bars_1m for 14 dates


## 5. Ticker Coverage Analysis

In [8]:
def check_ticker_coverage(date: str) -> dict:
    snapshot_tickers = set(
        r[0] for r in ch.query(
            "SELECT DISTINCT symbol FROM market_snapshot FINAL WHERE date = {d:String}",
            parameters={"d": date}
        ).result_rows
    )
    trades_tickers = set(
        r[0] for r in ch.query(
            "SELECT DISTINCT ticker FROM trades FINAL WHERE date = {d:String}",
            parameters={"d": date}
        ).result_rows
    )
    
    missing = snapshot_tickers - trades_tickers
    coverage = len(trades_tickers & snapshot_tickers) / len(snapshot_tickers) * 100 if snapshot_tickers else 0
    
    return {
        "date": date,
        "snapshot": len(snapshot_tickers),
        "trades": len(trades_tickers),
        "coverage": coverage,
        "missing_count": len(missing),
    }

coverage = [check_ticker_coverage(r['date']) for r in ready_dates]
pl.DataFrame(coverage)

shape: (0, 0)
┌┐
╞╡
└┘

## 6. Factor Value Preview

Preview trade_rate values to see if signals might trigger.

In [9]:
if ready_dates:
    test_date = ready_dates[0]['date']
    print(f"Previewing trade_rate for {test_date}")
    
    from jerry_trader.services.backtest.runner import BacktestRunner
    from jerry_trader.services.backtest.config import BacktestConfig
    from jerry_trader.services.backtest.batch_engine import FactorEngineBatchAdapter
    import logging
    logging.disable(logging.WARNING)
    
    config = BacktestConfig(date=test_date, rules_dir="config/rules/")
    runner = BacktestRunner(config, ch_client=ch)
    candidates = runner._step_prefilter(config)
    td_map = runner._step_load(candidates, config)
    
    engine = FactorEngineBatchAdapter()
    max_rates = []
    for sym, td in td_map.items():
        ts = engine.compute(td)
        if ts:
            rates = [v.get('trade_rate', 0) for v in ts.values()]
            max_rates.append((sym, max(rates) if rates else 0))
    
    max_rates.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\nTop 10 by max trade_rate (threshold=200):")
    for sym, rate in max_rates[:10]:
        trigger = "YES" if rate > 200 else "NO"
        print(f"  {sym:10s}: {rate:>8.1f}  [{trigger}]")
    
    n_trigger = sum(1 for _, r in max_rates if r > 200)
    print(f"\n{n_trigger}/{len(max_rates)} tickers have trade_rate > 200")
else:
    print("No ready dates")

No ready dates


---
**End of Report**